In [1]:
from pydantic_settings import BaseSettings
import re
import io

import pdfplumber
from docx import Document
from pydantic import BaseModel
import faiss
import numpy as np

from functools import lru_cache

import numpy as np
from fastembed import TextEmbedding

d:\Document\AI AGENT\resume-matcher\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")


def anonymize(text: str) -> str:
    """Remove contact details. Extend to strip names, gender, age, photos."""
    text = EMAIL.sub("[EMAIL]", text)
    return PHONE.sub("[PHONE]", text)

In [3]:
class ParsedResume(BaseModel):
    id: str
    text: str
    skills: list[str]
    years_experience: float


class MatchResult(BaseModel):
    resume_id: str
    score: float
    semantic: float
    skill_overlap: float
    matched_skills: list[str]
    missing_skills: list[str]

In [4]:

def hybrid_score(
    resume_id: str,
    semantic: float,
    resume_skills: list[str],
    job_skills: list[str],
    resume_years: float,
    required_years: float = 0.0,
) -> MatchResult:
    job_set, res_set = set(job_skills), set(resume_skills)
    matched = sorted(job_set & res_set)
    missing = sorted(job_set - res_set)
    overlap = len(matched) / len(job_set) if job_set else 0.0
    exp = min(resume_years / required_years, 1.0) if required_years else 1.0

    score = (
        settings.w_semantic * max(semantic, 0)
        + settings.w_skills * overlap
        + settings.w_experience * exp
    )
    return MatchResult(
        resume_id=resume_id,
        score=round(score, 4),
        semantic=round(semantic, 4),
        skill_overlap=round(overlap, 4),
        matched_skills=matched,
        missing_skills=missing,
    )


In [5]:

class VectorStore:
    """In-memory FAISS store. Swap for pgvector/Qdrant in production."""

    def __init__(self, dim: int = 384):
        self.index = faiss.IndexFlatIP(dim)  # cosine on normalized vectors
        self.ids: list[str] = []

    def add(self, rid: str, vec: np.ndarray) -> None:
        self.index.add(vec.reshape(1, -1))
        self.ids.append(rid)

    def search(self, vec: np.ndarray, k: int = 10) -> list[tuple[str, float]]:
        if not self.ids:
            return []
        k = min(k, len(self.ids))
        scores, idx = self.index.search(vec.reshape(1, -1), k)
        return [(self.ids[i], float(s)) for i, s in zip(idx[0], scores[0])]

In [6]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"          # use the regular HTTP download
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"   # longer timeout for slow networks
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


@lru_cache(maxsize=1)
def _model() -> TextEmbedding:
    return TextEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_dir=r"D:\Document\AI AGENT\resume-matcher\models_cache",
    )


def embed(texts: list[str]) -> np.ndarray:
    vecs = np.array(list(_model().embed(texts)), dtype="float32")
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / np.clip(norms, 1e-12, None)

In [7]:
# Starter list; replace with a larger taxonomy (e.g. ESCO) or LLM extraction.
SKILLS = [
    "python", "java", "javascript", "typescript", "react", "node.js", "sql",
    "postgresql", "mongodb", "docker", "kubernetes", "aws", "gcp", "azure",
    "machine learning", "deep learning", "nlp", "pytorch", "tensorflow",
    "fastapi", "django", "flask", "git", "linux", "pandas", "numpy",
]

YEARS = re.compile(r"(\d+(?:\.\d+)?)\+?\s*(?:years|yrs)", re.I)


def extract_skills(text: str) -> list[str]:
    low = text.lower()
    return sorted({s for s in SKILLS if re.search(rf"\b{re.escape(s)}\b", low)})


def extract_years(text: str) -> float:
    vals = [float(m) for m in YEARS.findall(text)]
    return max(vals) if vals else 0.0

In [8]:
class Settings(BaseSettings):
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    anthropic_api_key: str = ""
    w_semantic: float = 0.6
    w_skills: float = 0.3
    w_experience: float = 0.1

    class Config:
        env_file = ".env"


settings = Settings()

C:\Users\HP\AppData\Local\Temp\ipykernel_22088\22402653.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  class Settings(BaseSettings):


In [9]:
pdf_path = r"D:\Document\AI AGENT\resume-matcher\notebook\resume.pdf"
with open(pdf_path, "rb") as file:
    pdf_data = file.read()
    filename = pdf_path.split("\\")[-1]
    
filename    
    


'resume.pdf'

In [10]:
def extract_text(filename: str, data: bytes) -> str:
    name = filename.lower()
    if name.endswith(".pdf"):
        with pdfplumber.open(io.BytesIO(data)) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    if name.endswith(".docx"):
        doc = Document(io.BytesIO(data))
        return "\n".join(p.text for p in doc.paragraphs)
    if name.endswith(".txt"):
        return data.decode("utf-8", errors="ignore")
    raise ValueError("Unsupported file type. Use PDF, DOCX or TXT.")

In [11]:
extracted_text = extract_text(filename, pdf_data)
print(extracted_text)

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Satyam Kumar
 Patna  6200996072  https://www.linkedin.com/in/satyam-kumar-06369537b/  https://github.com/Satyam-Raaz
Skills Summary
MySQL and PostgreSQL database I'm currently studying for my Bachelor of Computer Application and have
a knack for Java, Spring, and Spring Boot Frameworks. I enjoy working
Spring Boot Framework
with JPA/Hibernate and MySQL/PostgreSQL databases, and I'm familiar
with REST APIs to ensure smooth data flow to frontend systems. I love
REST APIs
tackling real-world challenges and thrive in team settings, thanks to my
Java strong communication skills.
Experience
Interests
Practo
Playing Outdoor Games
Internship
Certifications Remote
1 June 2025 - 15 July 2025
Full Stack Development
Aimerz.ai I had a fantastic virtual internship at Practo Technologies Pvt. Ltd., where
July 2025 I got to build a Full Stack Web Application for ordering diagnostic tests
 https://pdflink.to/38ff8554/ through doctor prescription uploads. During this experience, I worked
with a var

In [12]:
text = anonymize(extracted_text)
print(text)


Satyam Kumar
 Patna  [PHONE]  https://www.linkedin.com/in/satyam-kumar-06369537b/  https://github.com/Satyam-Raaz
Skills Summary
MySQL and PostgreSQL database I'm currently studying for my Bachelor of Computer Application and have
a knack for Java, Spring, and Spring Boot Frameworks. I enjoy working
Spring Boot Framework
with JPA/Hibernate and MySQL/PostgreSQL databases, and I'm familiar
with REST APIs to ensure smooth data flow to frontend systems. I love
REST APIs
tackling real-world challenges and thrive in team settings, thanks to my
Java strong communication skills.
Experience
Interests
Practo
Playing Outdoor Games
Internship
Certifications Remote
1 June 2025 - 15 July 2025
Full Stack Development
Aimerz.ai I had a fantastic virtual internship at Practo Technologies Pvt. Ltd., where
July 2025 I got to build a Full Stack Web Application for ordering diagnostic tests
 https://pdflink.to/38ff8554/ through doctor prescription uploads. During this experience, I worked
with a variet

In [13]:
extracted_skills = extract_skills(text)
extracted_exp_years=extract_years(text)
print("Extracted Skills:", extracted_skills)
print("Extracted Years of Experience:", extracted_exp_years)

Extracted Skills: ['java', 'postgresql']
Extracted Years of Experience: 0.0


In [14]:
import uuid

rid = uuid.uuid4().hex[:8]
parsed = ParsedResume(
    id=rid,
    text=text,
    skills=extract_skills(text),
    years_experience=extract_years(text),
)
print(parsed)

id='c3623442' text="Satyam Kumar\n\ue316 Patna \ue3b8 [PHONE] \ue2e2 https://www.linkedin.com/in/satyam-kumar-06369537b/ \ue576 https://github.com/Satyam-Raaz\nSkills Summary\nMySQL and PostgreSQL database I'm currently studying for my Bachelor of Computer Application and have\na knack for Java, Spring, and Spring Boot Frameworks. I enjoy working\nSpring Boot Framework\nwith JPA/Hibernate and MySQL/PostgreSQL databases, and I'm familiar\nwith REST APIs to ensure smooth data flow to frontend systems. I love\nREST APIs\ntackling real-world challenges and thrive in team settings, thanks to my\nJava strong communication skills.\nExperience\nInterests\nPracto\nPlaying Outdoor Games\nInternship\nCertifications Remote\n1 June 2025 - 15 July 2025\nFull Stack Development\nAimerz.ai I had a fantastic virtual internship at Practo Technologies Pvt. Ltd., where\nJuly 2025 I got to build a Full Stack Web Application for ordering diagnostic tests\n\ue2e2 https://pdflink.to/38ff8554/ through doctor pr

In [15]:
resumes: dict[str, ParsedResume] = {}
resumes[rid] = parsed

In [16]:
print("Stored resumes:", resumes)

Stored resumes: {'c3623442': ParsedResume(id='c3623442', text="Satyam Kumar\n\ue316 Patna \ue3b8 [PHONE] \ue2e2 https://www.linkedin.com/in/satyam-kumar-06369537b/ \ue576 https://github.com/Satyam-Raaz\nSkills Summary\nMySQL and PostgreSQL database I'm currently studying for my Bachelor of Computer Application and have\na knack for Java, Spring, and Spring Boot Frameworks. I enjoy working\nSpring Boot Framework\nwith JPA/Hibernate and MySQL/PostgreSQL databases, and I'm familiar\nwith REST APIs to ensure smooth data flow to frontend systems. I love\nREST APIs\ntackling real-world challenges and thrive in team settings, thanks to my\nJava strong communication skills.\nExperience\nInterests\nPracto\nPlaying Outdoor Games\nInternship\nCertifications Remote\n1 June 2025 - 15 July 2025\nFull Stack Development\nAimerz.ai I had a fantastic virtual internship at Practo Technologies Pvt. Ltd., where\nJuly 2025 I got to build a Full Stack Web Application for ordering diagnostic tests\n\ue2e2 htt

In [17]:
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [18]:
store = VectorStore()
store.add(rid, embed([text])[0])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]d:\Document\AI AGENT\resume-matcher\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\Document\AI AGENT\resume-matcher\models_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not i

In [21]:
job_description = """Java Spring Boot Developer

Experience required: 3+ years

About the role
We are looking for a Java Spring Boot Developer to design, build, and maintain scalable backend services and REST APIs for our web and mobile applications. You will work with a cross-functional team to deliver secure, high-performance microservices.

Responsibilities

Develop and maintain RESTful APIs and microservices using Java and Spring Boot
Design database schemas and write optimized queries in PostgreSQL and MongoDB
Implement authentication and authorization with Spring Security and JWT
Write unit and integration tests (JUnit, Mockito)
Containerize applications with Docker and deploy them on Kubernetes and AWS
Participate in code reviews, using Git for version control and CI/CD pipelines
Troubleshoot production issues and improve application performance

Required skills

3+ years of experience in Java (8 or higher)
Strong hands-on experience with Spring Boot, Spring MVC, and Spring Data JPA
Experience building REST APIs and microservices
Good knowledge of SQL and relational databases such as PostgreSQL
Familiarity with Docker and Git
Understanding of Linux environments

Nice to have

Kubernetes and AWS experience
Kafka or RabbitMQ
MongoDB or Redis
Hibernate, Maven or Gradle

Education: Bachelor’s degree in Computer Science, IT, or a related field."""
job_skills = extract_skills(job_description)
print(job_skills)

['aws', 'docker', 'git', 'java', 'kubernetes', 'linux', 'mongodb', 'postgresql', 'sql']


In [22]:
hits = store.search(embed([job_description])[0], k=3)


In [23]:
required_years=extract_years(job_description)


In [24]:
results = [
        hybrid_score(rid, sim, resumes[rid].skills, job_skills,
                     resumes[rid].years_experience, required_years)
        for rid, sim in hits
]

In [25]:
print(sorted(results, key=lambda r: r.score, reverse=True))

[MatchResult(resume_id='c3623442', score=0.4386, semantic=0.6199, skill_overlap=0.2222, matched_skills=['java', 'postgresql'], missing_skills=['aws', 'docker', 'git', 'kubernetes', 'linux', 'mongodb', 'sql'])]


In [ ]:
print("Required Years of Experience:", required_years)

result = hybrid_score(rid, 0.5, resumes[rid].skills, job_skills,resumes[rid].years_experience, required_years)


Required Years of Experience: 3.0


In [ ]:
print(result)

resume_id='19a0b973' score=0.3667 semantic=0.5 skill_overlap=0.2222 matched_skills=['java', 'postgresql'] missing_skills=['aws', 'docker', 'git', 'kubernetes', 'linux', 'mongodb', 'sql']
